<a href="https://colab.research.google.com/github/Jozen1/EntregasGO_Equipo4/blob/main/notebooks/Avance1B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Clona el repositorio 'Operations-Management' de GitHub en el entorno actual.
!git clone https://github.com/LuisRojo-Gonzalez/Operations-Management.git

Cloning into 'Operations-Management'...
remote: Enumerating objects: 445, done.
remote: Counting objects: 100% (445/445), done.
remote: Compressing objects: 100% (437/437), done.
remote: Total 445 (delta 32), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (445/445), 15.49 MiB | 12.81 MiB/s, done.
Resolving deltas: 100% (32/32), done.


In [ ]:
# Muestra la ruta del directorio de trabajo actual.
!pwd

/content


In [ ]:
# Instalación de la librería necesaria (ejecutar en caso de solo ser necesario)
!pip install statsforecast

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 501.8/501.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.7/348.7 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 3.7 MB/s eta 0:00:00


In [ ]:
# Se importan las librerías necesarias
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, SeasonalNaive, AutoARIMA
import pandas as pd
import numpy as np

In [ ]:
# Tratamiento de censura y preparación
ventas = pd.read_csv('/content/Operations-Management/CaseStudy/Datos/demanda_semanal.csv', parse_dates=['fecha_lunes'])

In [ ]:
# Supuesto: La demanda es constante durante los 7 días de la semana.
# Reconstruimos la demanda compensando por los días en que no hubo disponibilidad.
ventas['demanda_real'] = ventas['ventas_cajas'] * (7 / (7 - ventas['dias_sin_stock']))

In [ ]:
# Preparar formato largo exigido por statsforecast (unique_id, ds, y)
df = ventas.rename(columns={'sku': 'unique_id', 'fecha_lunes': 'ds', 'demanda_real': 'y'})
df = df[['unique_id', 'ds', 'y']]

In [ ]:
# Se exige al menos benchmark ingenuo, ETS y SARIMA
# Frecuencia semanal ('W-MON'), temporada anual asumimos 52 semanas.
sf = StatsForecast(
    models=[
        SeasonalNaive(season_length=52),  # Benchmark obligatorio
        AutoETS(season_length=52),        # ETS automático
        AutoARIMA(season_length=52)       # SARIMA automático
    ],
    freq='W-MON',

    # Si ETS o ARIMA colapsan por falta de datos, usa el modelo ingenuo
    fallback_model=SeasonalNaive(season_length=52)
)

In [ ]:
# El caso exige al menos 8 ventanas (n_windows=8) y horizonte h=13
cv = sf.cross_validation(df=df, h=13, step_size=1, n_windows=8)

/usr/local/lib/python3.13/dist-packages/statsforecast/arima.py:1745: UserWarning: The chosen seasonal unit root test encountered an error when testing for the 0 difference.
From seas(): `mstl` cannot handle missing values. Please raise an issue to include this feature.
0 seasonal differences will be used. Consider using a different unit root test.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsforecast/arima.py:1745: UserWarning: The chosen seasonal unit root test encountered an error when testing for the 0 difference.
From seas(): `mstl` cannot handle missing values. Please raise an issue to include this feature.
0 seasonal differences will be used. Consider using a different unit root test.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsforecast/arima.py:1745: UserWarning: The chosen seasonal unit root test encountered an error when testing for the 0 difference.
From seas(): `mstl` cannot handle missing values. Please raise an issue to include this featu

In [ ]:
# Cálculo de wape por horizonte sin leakage
def wape(y, yhat):
    return 100 * np.sum(np.abs(y - yhat)) / np.sum(np.abs(y))

In [ ]:
# Calcular el horizonte 'h' para cada fila
dh = cv['ds'].dt.to_period('W') - cv['cutoff'].dt.to_period('W')
cv['h'] = dh.apply(lambda x: x.n)

In [ ]:
# Calcular WAPE agrupando por horizonte para evaluar degradación
tabla_wape = cv.groupby('h').apply(lambda g: pd.Series({
    'Naive': wape(g['y'], g['SeasonalNaive']),
    'ETS': wape(g['y'], g['AutoETS']),
    'ARIMA': wape(g['y'], g['AutoARIMA'])
}))

/tmp/ipykernel_1991/1100649769.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tabla_wape = cv.groupby('h').apply(lambda g: pd.Series({


In [ ]:
print("WAPE por horizonte (El error casi siempre CRECE con h): \n")
print(tabla_wape)

WAPE por horizonte (El error casi siempre CRECE con h): 

        Naive        ETS      ARIMA
h                                  
1   26.603142  24.562025  33.311404
2   26.256126  24.606760  35.201156
3   26.807878  24.254458  35.892881
4   28.772136  26.180390  36.331084
5   26.388912  24.868323  33.911031
6   26.283339  24.221436  34.121954
7   25.708393  21.457805  33.722708
8   25.911121  22.907988  34.400185
9   27.724809  25.459301  37.738775
10  28.635564  26.189849  36.925236
11  28.366765  26.204935  37.102873
12  26.488838  27.796476  38.725184
13  27.083127  26.637909  38.459964


hola